# Modelos de referencia oficiales de FlowNP en Colab

Entrena con el **código oficial** (`danrsm/flowNP`, commit fijado) los cuatro modelos de la comparación:
FlowNP oficial (`fnp`), proceso neuronal condicional (`cnp`), proceso neuronal atento (`anp`) y
proceso neuronal con transformador autorregresivo (`tnpa`), con 5 semillas.

**Reparto:** cada persona ejecuta un núcleo, en su propia cuenta de Colab y a la vez.

| Persona | Núcleo | Ejecuciones |
|---|---|---|
| 1 | exponencial cuadrático (`rbf`) | 4 modelos × 5 semillas = 20 |
| 2 | Matérn-5/2 (`matern`) | 20 |
| 3 | periódico (`periodic`) | 20 |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4*.

**Si Colab se desconecta:** reconecta y vuelve a ejecutar las celdas 1, 2, 3 y 5. Cada ejecución
reanuda desde su último punto de control (se guarda cada 1000 pasos en Drive); las terminadas se saltan.

## 1. Configuración — cambia solo `PERSONA`

In [ ]:
PERSONA = 1            # 1, 2 o 3

KERNEL_DE = {1: "rbf", 2: "matern", 3: "periodic"}
KERNELS   = [KERNEL_DE[PERSONA]]
MODELS    = ["fnp", "cnp", "anp", "tnpa"]
SEEDS     = [0, 1, 2, 3, 4]
NUM_STEPS = 100_000    # configuración oficial; no cambiar sin acordarlo entre las tres personas
PARALLEL  = 2          # ejecuciones simultáneas en la misma tarjeta gráfica (Colab gratuito tiene 2 procesadores)
POLL      = 60         # segundos entre informes de progreso
EXPNAME   = "main"
EXTRA     = []         # argumentos adicionales para gp.py (vacío = configuración oficial)

OFFICIAL_COMMIT = "f3403b6479dad33a7b93ba6edcf93c53d40f8e65"
DRIVE_DIR = "/content/drive/MyDrive/flownp_runs"   # carpeta compartida del grupo (ver README)
OUT  = f"{DRIVE_DIR}/official"
CODE = "/content/third_party/flowNP"
print("Persona", PERSONA, "-> núcleo", KERNELS, "| salida:", OUT)

## 2. Preparar el entorno

In [ ]:
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "No hay tarjeta gráfica: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4"
print("Tarjeta gráfica:", gpu)

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

if not os.path.isdir(CODE):
    !git clone -q https://github.com/danrsm/flowNP.git {CODE}
!git -C {CODE} checkout -q {OFFICIAL_COMMIT} && git -C {CODE} log -1 --format="Código oficial en el commit %h (%cd)"
!pip -q install attrdictionary addict prettytable torchdiffeq
!pip -q install --no-deps flow-matching==1.0.10

# El código oficial lee las rutas de utils/paths.py: se apuntan a Drive.
open(f"{CODE}/utils/paths.py", "w").write(f'''import os
datasets_path = "{OUT}/datasets"
output_path = "{OUT}"
evalsets_path = os.path.join(output_path, "evalsets")
results_path = os.path.join(output_path, "results")
''')
import torch; print("PyTorch", torch.__version__)

## 3. Conjunto de evaluación

Se genera **una sola vez por núcleo**, en procesador y con semilla 0, y se guarda en Drive. Todos los
modelos (y después nuestra reimplementación) se evalúan sobre este mismo fichero. Si ya existe, no se toca.

In [ ]:
import sys, argparse, hashlib
sys.path.insert(0, CODE); os.chdir(CODE)
import gp

REFERENCIA = {"rbf": "1635764c579b3f33", "matern": "059a9ba6662c6d7d", "periodic": "0f43ba32042fac80"}
for k in KERNELS:
    a = argparse.Namespace(eval_kernel=k, kernel=k, eval_ctx10=False, eval_seed=0, eval_num_batches=100,
                           eval_batch_size=128, max_num_points=50, device="cpu", eval_filename=None)
    path, fn = gp.get_eval_path(a)
    f = os.path.join(path, fn)
    if not os.path.isfile(f):
        gp.gen_evalset(a)
    h = hashlib.sha256()
    for b in torch.load(f, weights_only=False):
        for key in sorted(b.keys()):
            h.update(b[key].cpu().numpy().tobytes())
    h = h.hexdigest()[:16]
    print(f"{k}: {f}\n  huella {h}", "(igual que la de referencia)" if h == REFERENCIA[k]
          else "(distinta de la de referencia: no pasa nada, lo que importa es que todos usen este fichero)")

## 4. Piloto de tiempos (unos 5 minutos, recomendable la primera vez)

Entrena 600 pasos de cada modelo y estima cuánto tardará todo. No toca los resultados principales.

In [ ]:
import re, time, subprocess
tiempos = {}
for m in MODELS:
    pid = f"p{int(time.time())}"
    cmd = ["python", "gp.py", "--mode=train", f"--kernel={KERNELS[0]}", f"--model={m}", "--expname=piloto",
           f"--expid={pid}", "--num_steps=600", "--print_freq=300", "--eval_freq=1000000000",
           "--eval_num_batches=1", "--eval_filename=piloto", "--plot_batch_size=4", "--plot_num_samples=4"]
    r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
    d = f"{OUT}/results/gp/piloto/{KERNELS[0]}/{m}/{pid}"
    logs = [x for x in os.listdir(d) if x.startswith("train_")] if os.path.isdir(d) else []
    secs = [float(x) for x in re.findall(r"\(([\d.]+) secs\)", open(f"{d}/{logs[0]}").read())] if logs else []
    if r.returncode != 0 or len(secs) < 2:
        print(f"{m}: el piloto falló. Últimas líneas:\n" + "\n".join((r.stdout + r.stderr).splitlines()[-15:]))
        continue
    tiempos[m] = secs[-1] / 300            # segundos por paso, ya sin el arranque
    print(f"{m:5s} {1000 * tiempos[m]:6.1f} ms/paso -> {tiempos[m] * NUM_STEPS / 60:5.0f} min por ejecución")
total = sum(tiempos[m] * NUM_STEPS * len(SEEDS) for m in tiempos) / 3600
print(f"\nEntrenamiento total: {total:.1f} h de tarjeta gráfica; con PARALLEL={PARALLEL}, "
      f"unas {total / PARALLEL:.1f} h de reloj si el paralelismo escala bien (más la evaluación final de cada ejecución).")

## 5. Lanzar todas las ejecuciones

Deja esta celda corriendo y la pestaña abierta (el portátil sin suspender). Informa cada minuto del paso
en que va cada ejecución. Los registros completos quedan en `OUT/logs/`.

In [ ]:
import os, re, time, subprocess, shutil, json
import torch

def run_dir(model, kernel, seed):
    return f"{OUT}/results/gp/{EXPNAME}/{kernel}/{model}/s{seed}"

def status(model, kernel, seed):
    d = run_dir(model, kernel, seed)
    ck = f"{d}/ckpt.tar"
    if not os.path.isfile(ck):
        return "nueva"
    try:
        trained = torch.load(ck, map_location="cpu", weights_only=False).step > NUM_STEPS
    except Exception:
        return "nueva"                          # punto de control corrupto (corte mientras se escribía)
    ev = f"{d}/eval_{kernel}.log"
    evaluated = os.path.isfile(ev) and "tar_ll" in open(ev).read()
    if trained:
        return "terminada" if evaluated else "evaluar"
    return "reanudar"

def command(model, kernel, seed, st):
    mode = "eval" if st == "evaluar" else "train"   # entrenamiento completo sin evaluación: solo evaluar
    c = ["python", "gp.py", f"--mode={mode}", f"--kernel={kernel}", f"--model={model}",
         f"--expname={EXPNAME}", f"--expid=s{seed}", f"--train_seed={seed}",
         f"--num_steps={NUM_STEPS}", "--print_freq=1000", "--save_freq=1000",
         "--eval_freq=1000000000",              # sin evaluaciones intermedias: solo la final
         "--plot_batch_size=4", "--plot_num_samples=10"] + EXTRA
    if model == "anp":                          # como en el artículo: 8 muestras al entrenar, 200 al evaluar
        c += ["--train_num_samples=8", "--eval_num_samples=200"]
    if st == "reanudar":
        c += ["--resume=1"]
    return c

def last_step(model, kernel, seed):
    d = run_dir(model, kernel, seed)
    logs = sorted(f for f in os.listdir(d) if f.startswith("train_")) if os.path.isdir(d) else []
    if not logs:
        return 0
    steps = re.findall(r" step (\d+) ", open(f"{d}/{logs[-1]}").read())
    return int(steps[-1]) if steps else 0

def launch_all():
    jobs = [(m, k, s) for k in KERNELS for s in SEEDS for m in MODELS]   # semilla 0 de todos primero
    pending = [j for j in jobs if status(*j) != "terminada"]
    print(f"{len(jobs)} ejecuciones, {len(jobs) - len(pending)} ya terminadas, {len(pending)} por hacer")
    env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    running, tries = {}, {}
    t0 = time.time()
    while pending or running:
        while pending and len(running) < PARALLEL:
            j = pending.pop(0)
            st = status(*j)
            ck = f"{run_dir(*j)}/ckpt.tar"
            if st == "nueva" and os.path.isfile(ck):
                os.remove(ck)                       # punto de control ilegible: se empieza de cero
            os.makedirs(f"{OUT}/logs", exist_ok=True)
            log = open(f"{OUT}/logs/{j[1]}_{j[0]}_s{j[2]}.out", "a")
            running[j] = subprocess.Popen(command(*j, st),
                                          cwd=CODE, stdout=log, stderr=subprocess.STDOUT, env=env)
            print(f"[{time.strftime('%H:%M')}] lanza {j[1]} {j[0]} semilla {j[2]} ({st})")
        time.sleep(POLL)
        for j, p in list(running.items()):
            if p.poll() is not None:
                del running[j]
                ok = status(*j) == "terminada"
                tries[j] = tries.get(j, 0) + (0 if ok else 1)
                retry = not ok and tries[j] <= 2
                if retry:
                    pending.append(j)               # reintenta: reanuda desde el último punto de control
                print(f"[{time.strftime('%H:%M')}] {'OK   ' if ok else 'FALLO'} {j[1]} {j[0]} semilla {j[2]}"
                      + ("" if ok else f" (código {p.returncode}{', se reintenta' if retry else ''})"
                         f" -> mira {OUT}/logs/{j[1]}_{j[0]}_s{j[2]}.out"))
        if running:
            prog = ", ".join(f"{m}/s{s} {last_step(m, k, s)}" for (m, k, s) in running)
            print(f"   {(time.time() - t0) / 60:5.1f} min · en marcha: {prog}", flush=True)
    print("Hecho.")

def results_table():
    import pandas as pd
    rows = []
    for k in KERNELS:
        for m in MODELS:
            for s in SEEDS:
                f = f"{run_dir(m, k, s)}/eval_{k}.log"
                if os.path.isfile(f):
                    ll = re.findall(r"tar_ll (-?[\d.]+)", open(f).read())
                    if ll:
                        rows.append(dict(kernel=k, model=m, seed=s, tar_ll=float(ll[-1])))
    df = pd.DataFrame(rows)
    if df.empty:
        print("Aún no hay evaluaciones terminadas."); return df
    df.to_csv(f"{OUT}/results_{'_'.join(KERNELS)}.csv", index=False)
    return df.groupby(["kernel", "model"]).tar_ll.agg(["mean", "std", "count"]).round(3)

launch_all()

## 6. Resultados (se puede ejecutar en cualquier momento)

In [ ]:
results_table()